# 🔬 crdt-merge v0.3.0 — A100 Absolute Ceiling Test> **Runtime**: A100 GPU (80GB system RAM)> **Goal**: Find the exact breaking point of every module at extreme scale> **Modules**: Core merge, Strategies, Streaming, Delta, JSON, Dedup, VerificationSet runtime to **A100 GPU** before running.

In [ ]:
# Install crdt-merge v0.3.0 from PyPI!pip install crdt-merge==0.3.0 psutil 2>&1 | tail -3import crdt_mergeprint(f"crdt-merge version: {crdt_merge.__version__ if hasattr(crdt_merge, '__version__') else '0.3.0'}")import psutilram_gb = psutil.virtual_memory().total / (1024**3)print(f"System RAM: {ram_gb:.1f} GB")print(f"Available: {psutil.virtual_memory().available / (1024**3):.1f} GB")

In [ ]:
import time, gc, json, sys, tracemalloc, osfrom contextlib import contextmanagerfrom dataclasses import dataclass, field, asdictRESULTS = []def record(suite, test, scale, metric, value, unit, notes=""):    r = {"suite": suite, "test": test, "scale": scale,         "metric": metric, "value": round(value, 4), "unit": unit, "notes": notes}    RESULTS.append(r)    print(f"  [{suite}] {test} @ {scale:,}: {metric}={value:.4f} {unit} {notes}")@contextmanagerdef mem_track():    gc.collect()    tracemalloc.start()    baseline = tracemalloc.get_traced_memory()[0]    yield lambda: (tracemalloc.get_traced_memory()[1] - baseline) / (1024*1024)    tracemalloc.stop()def gen_rows(n, prefix="a", fields=5):    return [dict(_id=f"{prefix}_{i}", _ts=float(i),                 **{f"f{j}": f"v_{prefix}_{i}_{j}" for j in range(fields)}) for i in range(n)]def gen_conflict_pair(n, overlap=0.5):    olap = int(n * overlap)    uniq = n - olap    a = [{"_id": f"r_{i}", "_ts": float(i), "val": f"a_{i}", "score": i%100, "tags": f"t{i%10}"} for i in range(n)]    b = [{"_id": f"new_{i}", "_ts": float(i+n), "val": f"b_{i}", "score": (i+50)%100, "tags": f"t{(i+5)%10}"} for i in range(uniq)]    b += [{"_id": f"r_{i}", "_ts": float(i+n), "val": f"b_c_{i}", "score": (i+25)%100, "tags": f"t{(i+3)%10}"} for i in range(olap)]    return a, b# Row generators for streaming (memory-efficient)def row_generator(n, prefix="a", fields=5):    for i in range(n):        yield dict(_id=f"{prefix}_{i}", _ts=float(i),                   **{f"f{j}": f"v_{prefix}_{i}_{j}" for j in range(fields)})def sorted_gen(n, prefix, offset=0):    for i in range(n):        yield {"_id": f"r_{i:012d}", "_ts": float(i + offset), "val": f"{prefix}_{i}"}print("Infrastructure ready ✅")

In [ ]:
# SUITE 1: CORE merge() — Push to 5M+from crdt_merge import mergeimport psutilprint("=" * 60)print("SUITE 1: CORE merge() — Progressive Scale to OOM")print("=" * 60)scales = [100_000, 500_000, 1_000_000, 2_000_000, 3_000_000, 5_000_000]for n in scales:    gc.collect()    avail_gb = psutil.virtual_memory().available / (1024**3)    print(f"\n  Available RAM: {avail_gb:.1f} GB")        if avail_gb < 2.0:        record("core", "merge", n, "status", 0, "SKIPPED", f"Only {avail_gb:.1f}GB free")        break        a, b = gen_conflict_pair(n)        with mem_track() as peak:        t0 = time.perf_counter()        try:            result = merge(a, b, key="_id", timestamp_col="_ts")            elapsed = time.perf_counter() - t0            mb = peak()            tp = (len(a) + len(b)) / elapsed            record("core", "merge", n, "time_sec", elapsed, "s")            record("core", "merge", n, "throughput", tp, "rows/s")            record("core", "merge", n, "peak_mb", mb, "MB")            record("core", "merge", n, "output_rows", len(result), "rows")        except MemoryError:            record("core", "merge", n, "status", 0, "OOM", "MemoryError")            print(f"  *** OOM at {n:,} rows ***")            break        except Exception as e:            record("core", "merge", n, "status", 0, "ERROR", str(e)[:120])            break        del a, b    try: del result    except: pass    gc.collect()

In [ ]:
# SUITE 2: STRATEGIES — MergeSchema to 5M+from crdt_merge.strategies import MergeSchema, LWW, MaxWins, UnionSet, Priorityprint("=" * 60)print("SUITE 2: STRATEGIES — Schema Resolution to 5M+")print("=" * 60)schema = MergeSchema(default=LWW(), score=MaxWins(), tags=UnionSet(","),                     status=Priority(["draft","review","approved","published"]))scales = [100_000, 500_000, 1_000_000, 2_000_000, 5_000_000]for n in scales:    gc.collect()    avail_gb = psutil.virtual_memory().available / (1024**3)    if avail_gb < 2.0:        record("strategies", "resolve_row", n, "status", 0, "SKIPPED", f"Only {avail_gb:.1f}GB free")        break        ra = [{"_id": f"r_{i}", "_ts": float(i), "val": f"a_{i}", "score": i%100,           "tags": f"t{i%5}", "status": "draft"} for i in range(n)]    rb = [{"_id": f"r_{i}", "_ts": float(i+n), "val": f"b_{i}", "score": (i+50)%100,           "tags": f"t{(i+3)%5}", "status": "review"} for i in range(n)]        with mem_track() as peak:        t0 = time.perf_counter()        try:            merged = [schema.resolve_row(a, b, timestamp_col="_ts") for a, b in zip(ra, rb)]            elapsed = time.perf_counter() - t0            mb = peak()            record("strategies", "resolve_row", n, "time_sec", elapsed, "s")            record("strategies", "resolve_row", n, "throughput", n/elapsed, "rows/s")            record("strategies", "resolve_row", n, "peak_mb", mb, "MB")            assert merged[0]["status"] == "review"            record("strategies", "resolve_row", n, "correct", 1, "bool")        except MemoryError:            record("strategies", "resolve_row", n, "status", 0, "OOM"); break        except Exception as e:            record("strategies", "resolve_row", n, "status", 0, "ERROR", str(e)[:120]); break    del ra, rb, merged; gc.collect()

In [ ]:
# SUITE 3: STREAMING — The Main Event: merge_sorted_stream to 100Mfrom crdt_merge.streaming import merge_stream, merge_sorted_stream, StreamStatsprint("=" * 60)print("SUITE 3A: merge_stream — Batched Streaming to 5M")print("=" * 60)scales_batched = [100_000, 500_000, 1_000_000, 2_000_000, 5_000_000]for n in scales_batched:    gc.collect()    a, b = gen_conflict_pair(n)    with mem_track() as peak:        t0 = time.perf_counter()        try:            stats = StreamStats()            count = 0            for batch in merge_stream(a, b, key="_id", timestamp_col="_ts",                                      batch_size=10000, stats=stats):                count += len(batch)                # DON'T accumulate — just count (simulates stream-to-disk)            elapsed = time.perf_counter() - t0            mb = peak()            record("streaming", "merge_stream", n, "time_sec", elapsed, "s")            record("streaming", "merge_stream", n, "throughput", (len(a)+len(b))/elapsed, "rows/s")            record("streaming", "merge_stream", n, "peak_mb", mb, "MB")            record("streaming", "merge_stream", n, "output_rows", count, "rows")            record("streaming", "merge_stream", n, "batches", stats.batches_processed, "batches")        except MemoryError:            record("streaming", "merge_stream", n, "status", 0, "OOM"); break        except Exception as e:            record("streaming", "merge_stream", n, "status", 0, "ERROR", str(e)[:120]); break    del a, b; gc.collect()print("\n" + "=" * 60)print("SUITE 3B: merge_sorted_stream — CONSTANT MEMORY to 100M")print("=" * 60)scales_sorted = [100_000, 500_000, 1_000_000, 5_000_000, 10_000_000, 50_000_000, 100_000_000]for n in scales_sorted:    gc.collect()    with mem_track() as peak:        t0 = time.perf_counter()        try:            stats = StreamStats()            count = 0            for batch in merge_sorted_stream(                sorted_gen(n, "a", offset=0),                sorted_gen(n, "b", offset=n),                key="_id", timestamp_col="_ts",                batch_size=10000, stats=stats            ):                count += len(batch)            elapsed = time.perf_counter() - t0            mb = peak()            record("sorted_stream", "merge_sorted", n, "time_sec", elapsed, "s")            record("sorted_stream", "merge_sorted", n, "throughput", (n*2)/elapsed, "rows/s")            record("sorted_stream", "merge_sorted", n, "peak_mb", mb, "MB")            record("sorted_stream", "merge_sorted", n, "output_rows", count, "rows")        except MemoryError:            record("sorted_stream", "merge_sorted", n, "status", 0, "OOM"); break        except Exception as e:            record("sorted_stream", "merge_sorted", n, "status", 0, "ERROR", str(e)[:120]); break

In [ ]:
# SUITE 4: BATCH SIZE TUNING at 1M rowsfrom crdt_merge.streaming import merge_stream, StreamStatsprint("=" * 60)print("SUITE 4: BATCH SIZE TUNING @ 1M rows")print("=" * 60)N = 1_000_000a, b = gen_conflict_pair(N)for bs in [500, 1_000, 2_500, 5_000, 10_000, 25_000, 50_000, 100_000]:    gc.collect()    with mem_track() as peak:        t0 = time.perf_counter()        try:            stats = StreamStats()            ct = 0            for batch in merge_stream(a, b, key="_id", timestamp_col="_ts",                                      batch_size=bs, stats=stats):                ct += len(batch)            elapsed = time.perf_counter() - t0            mb = peak()            record("batch_tune", f"bs_{bs}", N, "time_sec", elapsed, "s")            record("batch_tune", f"bs_{bs}", N, "throughput", (len(a)+len(b))/elapsed, "rows/s")            record("batch_tune", f"bs_{bs}", N, "peak_mb", mb, "MB")            record("batch_tune", f"bs_{bs}", N, "batches", stats.batches_processed, "batches")        except Exception as e:            record("batch_tune", f"bs_{bs}", N, "status", 0, "ERROR", str(e)[:120])del a, b; gc.collect()

In [ ]:
# SUITE 5: DELTA SYNC — compute + apply to 2M+from crdt_merge.delta import compute_delta, apply_delta, compose_deltasprint("=" * 60)print("SUITE 5: DELTA SYNC — Extreme Scale")print("=" * 60)scales = [100_000, 500_000, 1_000_000, 2_000_000]for n in scales:    gc.collect()    avail_gb = psutil.virtual_memory().available / (1024**3)    if avail_gb < 3.0:        record("delta", "combined", n, "status", 0, "SKIPPED"); break        old = gen_rows(n, "old", 5)    new = list(old)    for i in range(n//3):        new[i] = {**new[i], "f0": f"MOD_{i}"}    for i in range(n//10):        new.append({"_id": f"added_{i}", "_ts": float(n+i), **{f"f{j}": f"n_{i}_{j}" for j in range(5)}})    new = new[n//10:]        with mem_track() as peak:        t0 = time.perf_counter()        try:            delta = compute_delta(old, new, key="_id", source_node="test")            elapsed_compute = time.perf_counter() - t0                        t1 = time.perf_counter()            result = apply_delta(old, delta, key="_id")            elapsed_apply = time.perf_counter() - t1            mb = peak()                        record("delta", "compute", n, "time_sec", elapsed_compute, "s")            record("delta", "compute", n, "throughput", n/elapsed_compute, "rows/s")            record("delta", "apply", n, "time_sec", elapsed_apply, "s")            record("delta", "apply", n, "throughput", n/elapsed_apply, "rows/s")            record("delta", "combined", n, "peak_mb", mb, "MB")        except MemoryError:            record("delta", "combined", n, "status", 0, "OOM"); break        except Exception as e:            record("delta", "combined", n, "status", 0, "ERROR", str(e)[:120]); break    del old, new; gc.collect()

In [ ]:
# SUITE 6: CRDT VERIFICATION — Proof at 1000 trialsfrom crdt_merge.verify import verify_crdtfrom crdt_merge import mergeprint("=" * 60)print("SUITE 6: CRDT PROPERTY VERIFICATION")print("=" * 60)counter = [0]def my_merge(a, b):    return merge(a, b, key="_id", timestamp_col="_ts")def gen_overlap():    counter[0] += 1    return [{"_id": f"r_{i}", "_ts": float(i + counter[0]*100),             "val": f"v{counter[0]}_{i}"} for i in range(50)]def set_eq(a, b):    return sorted(a, key=lambda r: r.get("_id","")) == sorted(b, key=lambda r: r.get("_id",""))for trials in [100, 500, 1000, 2000]:    gc.collect()    t0 = time.perf_counter()    result = verify_crdt(my_merge, gen_fn=gen_overlap, trials=trials, eq_fn=set_eq)    elapsed = time.perf_counter() - t0    all_ok = (result.commutativity.passed and result.associativity.passed and               result.idempotency.passed)    conv = result.convergence.passed if result.convergence else "N/A"    record("verify", f"trials_{trials}", trials, "time_sec", elapsed, "s")    record("verify", f"trials_{trials}", trials, "throughput", trials/elapsed, "trials/s")    record("verify", f"trials_{trials}", trials, "all_passed", 1 if all_ok else 0, "bool")    record("verify", f"trials_{trials}", trials, "convergence", 1 if conv else 0, "bool")    print(f"  C={result.commutativity.passed} A={result.associativity.passed} "          f"I={result.idempotency.passed} Conv={conv}")

In [ ]:
# SUITE 7: MULTI-NODE CONVERGENCE — 5 replicas at 500K+from crdt_merge import mergeprint("=" * 60)print("SUITE 7: MULTI-NODE CONVERGENCE — 5 Replicas")print("=" * 60)def set_eq(a, b):    return sorted(a, key=lambda r: r.get("_id","")) == sorted(b, key=lambda r: r.get("_id",""))scales = [10_000, 100_000, 500_000, 1_000_000]NUM_REPLICAS = 5for n in scales:    gc.collect()    avail_gb = psutil.virtual_memory().available / (1024**3)    if avail_gb < 5.0:        record("convergence", f"replicas_{NUM_REPLICAS}", n, "status", 0, "SKIPPED"); break        # Generate 5 replicas with partial overlap    replicas = []    for r in range(NUM_REPLICAS):        replica = [{"_id": f"r_{i}", "_ts": float(i + r*n),                    "val": f"rep{r}_{i}"} for i in range(n)]        # Each replica has 20% unique data        for i in range(n//5):            replica.append({"_id": f"uniq_{r}_{i}", "_ts": float(r*n + n + i),                           "val": f"unique_{r}_{i}"})        replicas.append(replica)        with mem_track() as peak:        t0 = time.perf_counter()        try:            # Merge all replicas pairwise: ((((R0 + R1) + R2) + R3) + R4)            merged_fwd = replicas[0]            for r in range(1, NUM_REPLICAS):                merged_fwd = merge(merged_fwd, replicas[r], key="_id", timestamp_col="_ts")                        # Reverse order: ((((R4 + R3) + R2) + R1) + R0)            merged_rev = replicas[-1]            for r in range(NUM_REPLICAS - 2, -1, -1):                merged_rev = merge(merged_rev, replicas[r], key="_id", timestamp_col="_ts")                        elapsed = time.perf_counter() - t0            mb = peak()                        converged = set_eq(merged_fwd, merged_rev)            record("convergence", f"replicas_{NUM_REPLICAS}", n, "time_sec", elapsed, "s")            record("convergence", f"replicas_{NUM_REPLICAS}", n, "peak_mb", mb, "MB")            record("convergence", f"replicas_{NUM_REPLICAS}", n, "converged", 1 if converged else 0, "bool")            record("convergence", f"replicas_{NUM_REPLICAS}", n, "output_rows", len(merged_fwd), "rows")            print(f"  {n:,} rows x {NUM_REPLICAS} replicas → converged={converged}, "                  f"output={len(merged_fwd):,} rows")        except MemoryError:            record("convergence", f"replicas_{NUM_REPLICAS}", n, "status", 0, "OOM"); break        except Exception as e:            record("convergence", f"replicas_{NUM_REPLICAS}", n, "status", 0, "ERROR", str(e)[:120]); break    del replicas; gc.collect()

In [ ]:
# SUITE 8: MEMORY PROFILE — Prove O(batch) vs O(n)from crdt_merge.streaming import merge_sorted_stream, StreamStatsimport tracemallocprint("=" * 60)print("SUITE 8: MEMORY PROFILE — O(batch) vs O(n) PROOF")print("=" * 60)# Test merge_sorted_stream at increasing scale, record PEAK memoryscales = [10_000, 100_000, 1_000_000, 5_000_000, 10_000_000]for n in scales:    gc.collect()    tracemalloc.start()    baseline = tracemalloc.get_traced_memory()[0]        t0 = time.perf_counter()    count = 0    peak_during = 0        for batch in merge_sorted_stream(        sorted_gen(n, "a", 0), sorted_gen(n, "b", n),        key="_id", timestamp_col="_ts", batch_size=10000    ):        count += len(batch)        current = tracemalloc.get_traced_memory()[1] - baseline        peak_during = max(peak_during, current)        elapsed = time.perf_counter() - t0    peak_mb = peak_during / (1024*1024)    tracemalloc.stop()        record("memory_proof", "sorted_stream", n, "time_sec", elapsed, "s")    record("memory_proof", "sorted_stream", n, "peak_mb", peak_mb, "MB")    record("memory_proof", "sorted_stream", n, "throughput", (n*2)/elapsed, "rows/s")    print(f"  {n:>12,} rows → {peak_mb:.2f} MB peak, {(n*2)/elapsed:,.0f} rows/s")print("\n  If memory stays flat → O(batch_size) PROVEN ✅")print("  If memory grows linearly → O(n) regression ❌")

In [ ]:
# FINAL REPORT — Save all resultsprint("=" * 60)print(f"COMPLETE: {len(RESULTS)} measurements collected")print("=" * 60)# Save JSON resultswith open("/content/crdt_merge_v030_a100_results.json", "w") as f:    json.dump(RESULTS, f, indent=2)# Print summary tableprint("\n### SUMMARY BY SUITE ###\n")suites = {}for r in RESULTS:    s = r["suite"]    if s not in suites:        suites[s] = {"count": 0, "max_scale": 0, "errors": 0}    suites[s]["count"] += 1    suites[s]["max_scale"] = max(suites[s]["max_scale"], r["scale"])    if r["metric"] == "status" and r["value"] == 0:        suites[s]["errors"] += 1for s, d in suites.items():    status = "✅" if d["errors"] == 0 else f"⚠️ {d['errors']} errors"    print(f"  {s:25s} | {d['count']:3d} measurements | max scale: {d['max_scale']:>12,} | {status}")# Print headline numbersprint("\n### HEADLINE NUMBERS ###\n")for r in RESULTS:    if r["metric"] == "throughput" and r["scale"] == max(x["scale"] for x in RESULTS if x["suite"] == r["suite"] and x["metric"] == "throughput"):        print(f"  {r['suite']:25s} @ {r['scale']:>12,}: {r['value']:>12,.0f} {r['unit']}")print(f"\nResults saved to /content/crdt_merge_v030_a100_results.json")print("Download and add to Nexus DB for permanent benchmarking.")# Copy to Drive if mountedtry:    from google.colab import drive    drive.mount('/content/drive')    import shutil    shutil.copy("/content/crdt_merge_v030_a100_results.json",                "/content/drive/MyDrive/crdt_merge_v030_a100_results.json")    print("✅ Results also saved to Google Drive")except:    print("Drive not mounted — results in /content/ only")